# 04 · XGBoost, image only and + morphometrics

Exactly the classical pipeline LR uses — the same flattened patch, the same two z-scored
morphometrics for `xgb_morph`, the same `StandardScaler` and train/test split — with the
estimator set to `xgboost.XGBClassifier`. Settings were **fixed in advance, not tuned**:
100 trees, depth 4, learning rate 0.3, 80% of patches and 10% of features per tree,
`hist` split finding with 32 bins, CPU, seed 95. The classes are balanced, so no class
weights. 7–16 min per model on the reference machine (≈ 70 min for all six).

`scripts/run_xgb.py` swaps the package's estimator factory for this process only; the
package itself is not edited. Saved models need `scripts/run_xgb_shim.py` importable.

**`RUN`** decides what this notebook does.
* With the original data on disk, `False` (default) reads the results of the run behind
  the manuscript and checks them; `True` re-trains into `runs/rerun`.
* On the synthetic data, `RUN` is `True`: there are no saved results to read, so the models
  are trained (into `runs/synthetic/rerun`; finished runs are skipped).

The manuscript's results folders are never overwritten, and nothing is written into the
dataset.

In [ ]:
import json, os, subprocess, sys, warnings
from pathlib import Path

# Find the repository folder (it holds scripts/paths.py) from wherever Jupyter was started.
HERE = Path.cwd()
while not (HERE / "scripts" / "paths.py").is_file() and HERE != HERE.parent:
    HERE = HERE.parent
sys.path.insert(0, str(HERE / "scripts"))
warnings.filterwarnings("ignore", category=FutureWarning)
# The histology images are not distributed. Without them this notebook runs on the
# synthetic dataset (scripts/make_synthetic_data.py) and writes to runs/synthetic/.
SYNTHETIC = (os.environ.get("VML_SYNTHETIC") == "1"          # forced, or no original data
             or not ((HERE / "data" / "Dataset_balanced" / "dataset.yaml").is_file() or "VML_DATA" in os.environ))
if SYNTHETIC:
    os.environ["VML_SYNTHETIC"] = "1"
import numpy as np
import pandas as pd
import paths as P
pd.set_option("display.width", 200, "display.max_columns", 30)
PY = sys.executable
if SYNTHETIC:   # generate and balance the synthetic data once (both steps skip existing output)
    subprocess.run([PY, str(HERE / "scripts" / "make_synthetic_data.py"), "--out", str(P.DATASETS)], check=True)
    subprocess.run([PY, str(HERE / "scripts" / "build_dataset.py"), "--only", "Dataset_balanced"],
                   check=True, capture_output=True)
print("mode           :", "SYNTHETIC data (not the manuscript's)" if SYNTHETIC else "original data")
print("dataset        :", P.DATA)
print("human data     :", P.HUMAN_ROOT)
print("results (used) :", P.RESULTS)

In [ ]:
RUN = SYNTHETIC                           # <- True: train into RERUN_BASE (always on synthetic data)
RERUN_BASE = P.WORK / "rerun"
BASE = RERUN_BASE if RUN else P.RESULTS   # where this notebook reads results from

In [ ]:
import run_xgb
RES = P.roots(BASE)["xgb"]
if RUN:
    run_xgb.main(RES)        # trains each task, then scores the human folders (resumable)
from manuscript_numbers import compute
N = compute(BASE, models=["xgb", "xgb_morph"], human=[])
pd.DataFrame(N["rat"]).T[["patch_f1_pos", "wsi_f1_pos", "patch_macro", "wsi_macro", "auc"]].round(3)